In [79]:
import re
import math
import random
import numpy as np
import pandas as pd
from collections import Counter
from typing import List, Dict, Tuple, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import gensim

In [80]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [81]:
import pathlib as path
import gensim

model_path = path.Path(r"C:\\ghoufrane\\Master\\M2\\llms\\Hanwritten-code-For-LLM-FineTuning\\models\\AraVec\\tweets_sg_100")
aravec_model = gensim.models.Word2Vec.load(str(model_path))

### new preprocess

In [82]:
# Loading raw data
train = pd.read_csv(r"..\\Notebooks\\data\\aravec_splits\\aravec_train.csv")
val = pd.read_csv(r"..\\Notebooks\\data\\aravec_splits\\aravec_val.csv")
test = pd.read_csv(r"..\\Notebooks\\data\\aravec_splits\\aravec_test.csv")

print(f"Raw splits loaded: Train={len(train)}, Val={len(val)}, Test={len(test)}")

Raw splits loaded: Train=11293, Val=1255, Test=3000


In [83]:
import pandas as pd
from sklearn.model_selection import train_test_split
BASE = "https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/"
full = pd.read_csv(BASE + "training_data.csv")
test = pd.read_csv(BASE + "testing_data.csv") # scellé jusqu'à la Partie F
full["id"], test["id"] = range(len(full)), range(len(test))

In [84]:
full.drop_duplicates(subset=["tweet"], keep=False, inplace=True)

In [85]:
def clean_str(text: str) -> str:
    """
    Official Arabic text normalization by AraVec creators (bakrianoo/aravec):
    1. Strips diacritics (tashkeel).
    2. Reduces repeated character elongations (tatweel).
    3. Normalizes Alef variants, Taa Marbuta, Alef Maksura, and punctuations.
    """
    if not isinstance(text, str):
        return ""
    
    search = ["أ", "إ", "آ", "ة", "_", "-", "/", ".", "،", " و ", " يا ", '"', "ـ", "'", "ى", "\\", '\n', '\t', '&quot;', '?', '؟', '!']
    replace = ["ا", "ا", "ا", "ه", " ", " ", "", "", "", " و", " يا", "", "", "", "ي", "", ' ', ' ', ' ', ' ? ', ' ؟ ', ' ! ']
    
    # 1. Remove tashkeel (diacritics)
    p_tashkeel = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    text = re.sub(p_tashkeel, "", text)
    
    # 2. Reduce elongation (tatweel/repetition)
    p_longation = re.compile(r'(.)\1+')
    subst = r"\1\1"
    text = re.sub(p_longation, subst, text)
    
    text = text.replace('وو', 'و')
    text = text.replace('يي', 'ي')
    text = text.replace('اا', 'ا')
    
    # 3. Replace characters 1-to-1
    for i in range(len(search)):
        text = text.replace(search[i], replace[i])
        
    return text.strip()


In [86]:
train['cleaned_tweet'] = train['tweet'].apply(clean_str)
val['cleaned_tweet'] = val['tweet'].apply(clean_str)

# Preview before and after
train[['tweet', 'cleaned_tweet']].head(3)


,tweet,cleaned_tweet
0,"""#لافروف: #موسكو تأمل بإجراء حوار بناء بشأن ال...",#لافروف: #موسكو تامل باجراء حوار بناء بشان الا...
1,"""RT @jalalyw: @EHSANFAKEEH تفصيل الخبر لمن يري...",RT @jalalyw: @EHSANFAKEEH تفصيل الخبر لمن يريد...
2,تحدثنا عن هذا الكتاب الجميل عن خروج المصريين ع...,تحدثنا عن هذا الكتاب الجميل عن خروج المصرين عل...


In [87]:
import emoji
from arabic_emojipedia.emoji_description import get_emoji_description

def replace_emoji_with_desc(text):
    if not isinstance(text, str):
        return ""
    
    # In emoji 1.x, the callback receives a match object:
    return emoji.replace_emoji(
        text, 
        replace=lambda m: f" {get_emoji_description(m.group())} "
    )

train["cleaned_tweet"] = train["tweet"].apply(replace_emoji_with_desc)
val["cleaned_tweet"] = val["tweet"].apply(replace_emoji_with_desc)


In [88]:
train.head(10)


,tweet,sarcasm,sentiment,dialect,id,cleaned_tweet
0,"""#لافروف: #موسكو تأمل بإجراء حوار بناء بشأن ال...",False,NEU,msa,2474,"""#لافروف: #موسكو تأمل بإجراء حوار بناء بشأن ال..."
1,"""RT @jalalyw: @EHSANFAKEEH تفصيل الخبر لمن يري...",False,NEU,msa,12107,"""RT @jalalyw: @EHSANFAKEEH تفصيل الخبر لمن يري..."
2,تحدثنا عن هذا الكتاب الجميل عن خروج المصريين ع...,False,NEU,msa,2135,تحدثنا عن هذا الكتاب الجميل عن خروج المصريين ع...
3,"""RT @Jokes__99: بوكيمون ولحقتوه عيل المسيح الد...",True,NEG,msa,640,"""RT @Jokes__99: بوكيمون ولحقتوه عيل المسيح الد..."
4,"""أوباما يلاحق محمد بن سلمان !!: https://t.co/x...",False,NEU,msa,6766,"""أوباما يلاحق محمد بن سلمان !!: https://t.co/x..."
5,"""تبرعات الهلال الاحمر الكويتي لن تسلم للهلال ا...",False,NEU,msa,5267,"""تبرعات الهلال الاحمر الكويتي لن تسلم للهلال ا..."
6,"""@riyadhalasaad عندما نشاهد مايجري في سوريا من...",False,NEG,msa,1318,"""@riyadhalasaad عندما نشاهد مايجري في سوريا من..."
7,"""يوسف سونة : دونالد ترامب و هيلاري كلينتون من ...",False,NEU,msa,10003,"""يوسف سونة : دونالد ترامب و هيلاري كلينتون من ..."
8,بموت ضحك عطو فرصه لغيركم يحزن,False,NEG,msa,4557,بموت ضحك عطو فرصه لغيركم يحزن
9,في قلبي رجل M❤️ يلغ...,False,POS,msa,49,في قلبي رجل M قلب أحمر ...


In [89]:
from tarfile import tar_filter
import re

def remove_punctuation(text):
    text = re.sub(r'[^\w\s ! ?]','', text)
    text = re.sub(r'@','', text)
    text = re.sub(r'#','', text)
    text = re.sub(r'http\S+','', text)
    text = re.sub(r'www\.\S+','', text)
    text = re.sub(r'_',' ', text)
    return text

train['tweet'] = train['tweet'].apply(remove_punctuation)
val['tweet'] = val['tweet'].apply(remove_punctuation)

train['tweet'].head(10)

0    لافروف موسكو تأمل بإجراء حوار بناء بشأن الأزمة...
1    RT jalalyw EHSANFAKEEH تفصيل الخبر لمن يريدالإ...
2    تحدثنا عن هذا الكتاب الجميل عن خروج المصريين ع...
3    RT Jokes  99 بوكيمون ولحقتوه عيل المسيح الدجال...
4           أوباما يلاحق محمد بن سلمان !!  via YouTube
5    تبرعات الهلال الاحمر الكويتي لن تسلم للهلال ال...
6    riyadhalasaad عندما نشاهد مايجري في سوريا من ق...
7    يوسف سونة  دونالد ترامب و هيلاري كلينتون من يح...
8                        بموت ضحك عطو فرصه لغيركم يحزن
9    في قلبي رجل M                            يلغي ...
Name: tweet, dtype: str

In [90]:
def delete_latin(text):
    latin_words = re.findall(r'[a-zA-Z0-9]+', text)

    for la in latin_words:
        if la in text:
            text = re.sub(rf'\b{re.escape(la)}\b'," ", text)
    return text

train["cleaned_tweet"] = train["tweet"].apply(delete_latin)
val["cleaned_tweet"] = val["tweet"].apply(delete_latin)


In [91]:
train.head(10)

,tweet,sarcasm,sentiment,dialect,id,cleaned_tweet
0,لافروف موسكو تأمل بإجراء حوار بناء بشأن الأزمة...,False,NEU,msa,2474,لافروف موسكو تأمل بإجراء حوار بناء بشأن الأزمة...
1,RT jalalyw EHSANFAKEEH تفصيل الخبر لمن يريدالإ...,False,NEU,msa,12107,تفصيل الخبر لمن يريدالإستزادة والإطلاع 1...
2,تحدثنا عن هذا الكتاب الجميل عن خروج المصريين ع...,False,NEU,msa,2135,تحدثنا عن هذا الكتاب الجميل عن خروج المصريين ع...
3,RT Jokes 99 بوكيمون ولحقتوه عيل المسيح الدجال...,True,NEG,msa,640,بوكيمون ولحقتوه عيل المسيح الدجال عميان...
4,أوباما يلاحق محمد بن سلمان !! via YouTube,False,NEU,msa,6766,أوباما يلاحق محمد بن سلمان !!
5,تبرعات الهلال الاحمر الكويتي لن تسلم للهلال ال...,False,NEU,msa,5267,تبرعات الهلال الاحمر الكويتي لن تسلم للهلال ال...
6,riyadhalasaad عندما نشاهد مايجري في سوريا من ق...,False,NEG,msa,1318,عندما نشاهد مايجري في سوريا من قتل للإطفال و...
7,يوسف سونة دونالد ترامب و هيلاري كلينتون من يح...,False,NEU,msa,10003,يوسف سونة دونالد ترامب و هيلاري كلينتون من يح...
8,بموت ضحك عطو فرصه لغيركم يحزن,False,NEG,msa,4557,بموت ضحك عطو فرصه لغيركم يحزن
9,في قلبي رجل M يلغي ...,False,POS,msa,49,في قلبي رجل يلغي ...


## Tokenization

In [92]:
def tokenize(text):
    if not isinstance(text,str):
        return []
    return text.strip().split()

train['tokens'] = train['cleaned_tweet'].apply(tokenize)

In [93]:
MIN_FREQ = 2
word_counts = Counter(word for tokens in train['tokens'] for word in tokens)

# Build word2idx with special tokens + words with frequency >= 2
word2idx = {'<PAD>': 0, '<UNK>': 1}
for word, count in word_counts.items():
    if count >= MIN_FREQ:
        word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}

print(f"Final Vocab Size: {len(word2idx):,} (Kept: {len(word2idx)-2:,}, Dropped: {sum(c < MIN_FREQ for c in word_counts.values()):,})")


Final Vocab Size: 15,575 (Kept: 15,573, Dropped: 31,098)


In [94]:
vocab_size = len(word2idx)
embed_dim = aravec_model.wv.vector_size  # 100 dimensions
print(f"Vocabulaire : {vocab_size} mots | Dimension des vecteurs AraVec : {embed_dim}")

Vocabulaire : 15575 mots | Dimension des vecteurs AraVec : 100


In [95]:
weights_matrix = np.random.normal(scale=0.02, size=(vocab_size, embed_dim)).astype(np.float32)
weights_matrix[0] = np.zeros(embed_dim, dtype=np.float32)

In [96]:
matched = 0
missing_words = []

In [97]:
for word, idx in word2idx.items():
    if idx == 0:  # Skip <PAD>
        continue
    cleaned = clean_str(word)
    
    # Priority: normalized word -> raw word
    if cleaned in aravec_model.wv:
        weights_matrix[idx] = aravec_model.wv[cleaned]
        matched += 1
    elif word in aravec_model.wv:
        weights_matrix[idx] = aravec_model.wv[word]
        matched += 1
    else:
        missing_words.append(word)

# 3. Summary & Coverage
total_words = vocab_size - 1  # Excluding <PAD>
coverage = (matched / total_words) * 100
print(f"--- AraVec Coverage Report ---")
print(f"• Found in AraVec   : {matched:,} / {total_words:,} ({coverage:.2f}%)")
print(f"• Missing (Random)  : {len(missing_words):,} ({100 - coverage:.2f}%)")

--- AraVec Coverage Report ---
• Found in AraVec   : 14,769 / 15,574 (94.83%)
• Missing (Random)  : 805 (5.17%)


In [98]:
weights_tensor = torch.tensor(weights_matrix, dtype=torch.float32)

### Embeddings

In [99]:
def tokens_to_ids(tokens):
    return [word2idx.get(w, 1) for w in tokens]  # 1 is <UNK>

train['token_ids'] = train['tokens'].apply(tokens_to_ids)



In [100]:
embed_dim = aravec_model.wv.vector_size  # 100

# 1. Random Embedding (trained from scratch)
embedding_random = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
nn.init.normal_(embedding_random.weight, mean=0.0, std=0.02)
embedding_random.weight.data[0].fill_(0.0)

# 2. AraVec Figée (Frozen)
embedding_aravec_frozen = nn.Embedding.from_pretrained(
    weights_tensor, 
    freeze=True, 
    padding_idx=0
)

# 3. AraVec Dégelée (Unfrozen / Trainable)
embedding_aravec_unfrozen = nn.Embedding.from_pretrained(
    weights_tensor.clone(), 
    freeze=False, 
    padding_idx=0
)


### BiLSTM CLASSIFIER

In [101]:
class BiLSTMClassifier(nn.Module):
    """
    C1: Bidirectional LSTM with choice of pooling (max-pooling or last hidden state).
    """
    def __init__(
        self,
        embedding_layer: nn.Embedding,
        hidden_dim: int = 128,
        num_layers: int = 1,
        dropout: float = 0.3,
        pooling: str = "max",      # "max" or "last"
        padding_idx: int = 0
    ):
        super().__init__()
        self.embedding = embedding_layer
        self.embed_dim = embedding_layer.embedding_dim
        self.hidden_dim = hidden_dim
        self.pooling = pooling
        self.padding_idx = padding_idx
        # Bidirectional LSTM: output per step will be 2 * hidden_dim
        self.lstm = nn.LSTM(
            input_size=self.embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch_size, seq_len]
        embeds = self.dropout(self.embedding(x))   # [batch_size, seq_len, embed_dim]
        lstm_out, (hn, cn) = self.lstm(embeds)     # lstm_out: [batch_size, seq_len, 2 * hidden_dim]
        if self.pooling == "max":
            # Mask out padding tokens so they don't affect max-pooling
            mask = (x != self.padding_idx).unsqueeze(-1)  # [batch_size, seq_len, 1]
            masked_out = lstm_out.masked_fill(~mask, -1e9)
            pooled, _ = torch.max(masked_out, dim=1)      # [batch_size, 2 * hidden_dim]
        elif self.pooling == "last":
            # Concat forward final hidden state and backward final hidden state
            # hn shape: [num_layers * 2, batch_size, hidden_dim]
            forward_last = hn[-2]   # [batch_size, hidden_dim]
            backward_last = hn[-1]  # [batch_size, hidden_dim]
            pooled = torch.cat([forward_last, backward_last], dim=-1)
        logits = self.fc(self.dropout(pooled)).squeeze(-1)  # [batch_size]
        return logits

### Bilstm + Attention Pooling

In [102]:
class BiLSTMWithAttention(nn.Module):
    """
    C2: Bidirectional LSTM with custom attention pooling.
    alpha_i = softmax(v^T tanh(W h_i)), c = sum(alpha_i * h_i)
    """
    def __init__(
        self,
        embedding_layer: nn.Embedding,
        hidden_dim: int = 128,
        attn_dim: int = 64,
        num_layers: int = 1,
        dropout: float = 0.3,
        padding_idx: int = 0
    ):
        super().__init__()
        self.embedding = embedding_layer
        self.embed_dim = embedding_layer.embedding_dim
        self.padding_idx = padding_idx

        # BiLSTM encoder
        self.lstm = nn.LSTM(
            input_size=self.embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        # Attention projection layers: W and v
        self.W = nn.Linear(hidden_dim * 2, attn_dim)
        self.v = nn.Linear(attn_dim, 1, bias=False)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x: torch.Tensor, return_attention: bool = False):
        # x: [batch_size, seq_len]
        embeds = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(embeds)  # [batch_size, seq_len, 2 * hidden_dim]

        # 1. Compute unnormalized attention scores: v^T tanh(W h_i)
        u = torch.tanh(self.W(lstm_out))           # [batch_size, seq_len, attn_dim]
        scores = self.v(u).squeeze(-1)             # [batch_size, seq_len]

        # 2. Mask padding positions so padding gets 0 attention weight
        pad_mask = (x == self.padding_idx)
        scores = scores.masked_fill(pad_mask, -1e9)

        # 3. Normalized attention weights: alpha = softmax(scores)
        alpha = F.softmax(scores, dim=-1)          # [batch_size, seq_len]

        # 4. Context vector: c = sum_i alpha_i * h_i
        # Batch matrix multiplication: [batch, 1, seq_len] @ [batch, seq_len, 2*H]
        context = torch.bmm(alpha.unsqueeze(1), lstm_out).squeeze(1)  # [batch, 2 * hidden_dim]

        logits = self.fc(self.dropout(context)).squeeze(-1)          # [batch_size]

        if return_attention:
            return logits, alpha
        return logits


### Mini Encoder Transformer

In [103]:
import math

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch_size, seq_len, d_model]
        return x + self.pe[:, :x.size(1), :]


class MiniTransformerClassifier(nn.Module):
    """
    C3: Mini Transformer Encoder (2 layers, d_model=128, 4 heads, sinusoidal PE, padding mask).
    Ablation parameter: use_pe (set False to test without positional encoding).
    """
    def __init__(
        self,
        embedding_layer: nn.Embedding,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 2,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
        use_pe: bool = True,          # Ablation toggle
        padding_idx: int = 0
    ):
        super().__init__()
        self.embedding = embedding_layer
        self.embed_dim = embedding_layer.embedding_dim
        self.d_model = d_model
        self.use_pe = use_pe
        self.padding_idx = padding_idx

        # Project embedding dimension (e.g. 100 for AraVec) to d_model (128)
        self.input_proj = nn.Linear(self.embed_dim, d_model) if self.embed_dim != d_model else nn.Identity()

        if self.use_pe:
            self.pos_encoder = SinusoidalPositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch_size, seq_len]
        # Padding mask: True where tokens are padding (PyTorch Transformer convention)
        src_key_padding_mask = (x == self.padding_idx)

        embeds = self.embedding(x)               # [batch, seq_len, embed_dim]
        out = self.input_proj(embeds)            # [batch, seq_len, d_model]

        if self.use_pe:
            out = self.pos_encoder(out)

        out = self.dropout(out)
        encoded = self.transformer_encoder(out, src_key_padding_mask=src_key_padding_mask)

        # Masked average pooling across sequence length (ignoring <PAD>)
        mask = (~src_key_padding_mask).unsqueeze(-1)  # [batch, seq_len, 1]
        sum_pooled = (encoded * mask).sum(dim=1)
        valid_lengths = mask.sum(dim=1).clamp(min=1)
        pooled = sum_pooled / valid_lengths           # [batch, d_model]

        logits = self.fc(self.dropout(pooled)).squeeze(-1)  # [batch_size]
        return logits


### Train and eval function

In [104]:
# Convert val and test using TRAIN's vocabulary (unseen words safely become <UNK> = 1)
val['token_ids']  = val['cleaned_tweet'].apply(lambda s: [word2idx.get(w, 1) for w in s.split()])
test['token_ids'] = test['tweet'].apply(lambda s: [word2idx.get(w, 1) for w in s.split()])

In [105]:
MAX_LEN = 60  # Covers ~98% of tweets

def pad_sequences_to_tensor(token_ids_series, max_len=MAX_LEN):
    padded = np.zeros((len(token_ids_series), max_len), dtype=np.int64)
    for i, ids in enumerate(token_ids_series):
        trunc = ids[:max_len]
        padded[i, :len(trunc)] = trunc
    return torch.tensor(padded, dtype=torch.long)

# Simple 2D Tensors
X_train = pad_sequences_to_tensor(train['token_ids']).to(DEVICE)
y_train = torch.tensor(train['sarcasm'].astype(int).values, dtype=torch.float32).to(DEVICE)

X_val   = pad_sequences_to_tensor(val['token_ids']).to(DEVICE)
y_val   = torch.tensor(val['sarcasm'].astype(int).values, dtype=torch.float32).to(DEVICE)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")


X_train shape: torch.Size([11293, 60]), y_train shape: torch.Size([11293])


In [106]:
def train_simple(
    model: nn.Module,
    X_tr: torch.Tensor,
    y_tr: torch.Tensor,
    X_v: torch.Tensor,
    y_v: torch.Tensor,
    criterion: nn.Module,
    epochs: int = 12,
    batch_size: int = 64,
    lr: float = 1e-3,
    patience: int = 4,
    find_best_threshold: bool = False
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    n_samples = len(X_tr)
    best_val_f1 = -1.0
    best_weights = None
    best_metrics = {}
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        # 1. Shuffle indices each epoch
        indices = torch.randperm(n_samples)
        model.train()

        # 2. Manual batch slice loop (No DataLoader)
        for i in range(0, n_samples, batch_size):
            batch_idx = indices[i : i + batch_size]
            x_b, y_b = X_tr[batch_idx], y_tr[batch_idx]

            optimizer.zero_grad()
            logits = model(x_b)
            loss = criterion(logits, y_b)
            loss.backward()
            optimizer.step()

        # 3. Validation
        model.eval()
        with torch.no_grad():
            val_logits = model(X_v)
            val_probs = torch.sigmoid(val_logits).cpu().numpy()
            y_v_np = y_v.cpu().numpy()

        # 4. Threshold check
        th = 0.5
        if find_best_threshold:
            th = max(
                np.arange(0.2, 0.8, 0.05),
                key=lambda t: f1_score(y_v_np, (val_probs >= t).astype(int), pos_label=1, zero_division=0)
            )

        val_pred = (val_probs >= th).astype(int)
        f1_sarcasm = f1_score(y_v_np, val_pred, pos_label=1, zero_division=0)

        # 5. Early Stopping
        if f1_sarcasm > best_val_f1:
            best_val_f1 = f1_sarcasm
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_metrics = {
                "F1_Sarcasm": round(f1_sarcasm, 4),
                "PR_AUC": round(average_precision_score(y_v_np, val_probs), 4),
                "Macro_F1": round(f1_score(y_v_np, val_pred, average='macro', zero_division=0), 4),
                "Threshold": round(th, 2)
            }
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(best_weights)
    return model, best_metrics


In [107]:
def build_model(arch: str, emb_mode: str) -> nn.Module:
    """Instantiates a model with the requested architecture and embedding mode."""
    emb = get_embedding_layer(emb_mode)
    
    if arch == "C1_BiLSTM_Max":
        return BiLSTMClassifier(emb, pooling="max")
    elif arch == "C1_BiLSTM_Last":
        return BiLSTMClassifier(emb, pooling="last")
    elif arch == "C2_BiLSTM_Attention":
        return BiLSTMWithAttention(emb)
    elif arch == "C3_Transformer_PE":
        return MiniTransformerClassifier(emb, use_pe=True)
    elif arch == "C3_Transformer_NoPE":  # Ablation without PE
        return MiniTransformerClassifier(emb, use_pe=False)
    else:
        raise ValueError(f"Unknown architecture: {arch}")


In [108]:
vocab_size = len(word2idx)
embed_dim = aravec_model.wv.vector_size  # 100

def get_embedding_layer(mode: str = "frozen") -> nn.Embedding:
    """
    Returns a fresh embedding layer for each experiment:
    1. 'random'   : Random weights trained from scratch
    2. 'frozen'   : AraVec weights locked (freeze=True)
    3. 'unfrozen' : AraVec weights fine-tuned (freeze=False)
    """
    if mode == "random":
        emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        nn.init.normal_(emb.weight, mean=0.0, std=0.02)
        emb.weight.data[0].fill_(0.0)
        return emb

    elif mode == "frozen":
        return nn.Embedding.from_pretrained(
            weights_tensor.clone(), 
            freeze=True, 
            padding_idx=0
        )

    elif mode == "unfrozen":
        # .clone() ensures each run starts from original AraVec weights
        return nn.Embedding.from_pretrained(
            weights_tensor.clone(), 
            freeze=False, 
            padding_idx=0
        )
    else:
        raise ValueError(f"Unknown embedding mode: {mode}")


In [109]:
import itertools

# Define what you want to compare
architectures   = ["C1_BiLSTM_Max", "C2_BiLSTM_Attention", "C3_Transformer_PE", "C3_Transformer_NoPE"]
embedding_modes = ["random", "frozen", "unfrozen"]

results = []
standard_loss = nn.BCEWithLogitsLoss()

print(f"Total experiments to run: {len(architectures) * len(embedding_modes)}\n" + "="*50)

for arch, emb_mode in itertools.product(architectures, embedding_modes):
    print(f"\n▶ Running: [{arch}] with Embedding [{emb_mode.upper()}]...")
    
    # 1. Instantiate fresh model
    model = build_model(arch, emb_mode).to(DEVICE)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # 2. Train and evaluate
    _, metrics = train_simple(
        model=model,
        X_tr=X_train,
        y_tr=y_train,
        X_v=X_val,
        y_v=y_val,
        criterion=standard_loss,
        epochs=12,
        batch_size=64,
        patience=4
    )

    # 3. Log identifiers
    metrics["Architecture"]    = arch
    metrics["Embedding_Mode"]  = emb_mode
    metrics["Trainable_Params"] = trainable_params

    results.append(metrics)

# 4. Display Comparison Table
df_comparison = pd.DataFrame(results)

# Reorder columns for clean presentation
display_cols = [
    "Architecture", 
    "Embedding_Mode", 
    "F1_Sarcasm", 
    "PR_AUC", 
    "Macro_F1", 
    "Trainable_Params", 
]

df_sorted = df_comparison[display_cols].sort_values(by="F1_Sarcasm", ascending=False)
display(df_sorted)


Total experiments to run: 12

▶ Running: [C1_BiLSTM_Max] with Embedding [RANDOM]...

▶ Running: [C1_BiLSTM_Max] with Embedding [FROZEN]...

▶ Running: [C1_BiLSTM_Max] with Embedding [UNFROZEN]...

▶ Running: [C2_BiLSTM_Attention] with Embedding [RANDOM]...

▶ Running: [C2_BiLSTM_Attention] with Embedding [FROZEN]...

▶ Running: [C2_BiLSTM_Attention] with Embedding [UNFROZEN]...

▶ Running: [C3_Transformer_PE] with Embedding [RANDOM]...

▶ Running: [C3_Transformer_PE] with Embedding [FROZEN]...

▶ Running: [C3_Transformer_PE] with Embedding [UNFROZEN]...

▶ Running: [C3_Transformer_NoPE] with Embedding [RANDOM]...

▶ Running: [C3_Transformer_NoPE] with Embedding [FROZEN]...

▶ Running: [C3_Transformer_NoPE] with Embedding [UNFROZEN]...


,Architecture,Embedding_Mode,F1_Sarcasm,PR_AUC,Macro_F1,Trainable_Params
11,C3_Transformer_NoPE,unfrozen,0.5290,0.5545,0.7157,1835517
8,C3_Transformer_PE,unfrozen,0.5173,0.5243,0.7083,1835517
10,C3_Transformer_NoPE,frozen,0.5160,0.5689,0.7153,278017
5,C2_BiLSTM_Attention,unfrozen,0.5035,0.4683,0.7000,1809789
6,C3_Transformer_PE,random,0.4638,0.4660,0.6680,1835517
2,C1_BiLSTM_Max,unfrozen,0.4558,0.5005,0.6837,1793277
7,C3_Transformer_PE,frozen,0.4535,0.5446,0.6833,278017
9,C3_Transformer_NoPE,random,0.4389,0.4440,0.6526,1835517
1,C1_BiLSTM_Max,frozen,0.4244,0.4644,0.6613,235777
0,C1_BiLSTM_Max,random,0.4230,0.4323,0.6595,1793277


### validation

In [110]:

def train_model(
    model, 
    X_tr, 
    y_tr, 
    X_v, 
    y_v, 
    criterion=None, 
    epochs=10, 
    batch_size=32, 
    lr=1e-3, 
    weight_decay=1e-4, 
    patience=4,
    eval_threshold=0.5
):
    """
    Entraîne le modèle avec une loss donnée et early stopping basé sur le Val F1-sarc.
    """
    if criterion is None:
        criterion = nn.BCEWithLogitsLoss()
        
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay
    )
    
    n_train = len(X_tr)
    n_val = len(X_v)
    best_val_f1 = -1.0
    best_weights = None
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        model.train()
        permutation = torch.randperm(n_train)
        total_train_loss = 0.0
        n_train_batches = 0

        for i in range(0, n_train, batch_size):
            idx = permutation[i:i + batch_size]
            bx = X_tr[idx].to(DEVICE)
            by = y_tr[idx].to(DEVICE)

            optimizer.zero_grad()
            logits = model(bx)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            n_train_batches += 1

        train_loss = total_train_loss / n_train_batches

        # Évaluation sur validation
        model.eval()
        total_val_loss = 0.0
        n_val_batches = 0
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for i in range(0, n_val, batch_size):
                bx = X_v[i:i + batch_size].to(DEVICE)
                by = y_v[i:i + batch_size].to(DEVICE)

                logits = model(bx)
                loss = criterion(logits, by)
                total_val_loss += loss.item()
                n_val_batches += 1

                preds = (torch.sigmoid(logits) >= eval_threshold).long()
                all_preds.extend(preds.cpu().tolist())
                all_targets.extend(by.cpu().long().tolist())

        val_loss = total_val_loss / n_val_batches
        val_f1 = f1_score(all_targets, all_preds, pos_label=1, zero_division=0)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_weights is not None:
        model.load_state_dict(best_weights)

    return model, best_val_f1


def evaluate_predictions(model, X_eval, y_eval, threshold=0.5, batch_size=32):
    """
    Calcule F1-sarc, Précision, Rappel et Macro-F1 à un seuil donné.
    """
    model.eval()
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for i in range(0, len(X_eval), batch_size):
            bx = X_eval[i:i + batch_size].to(DEVICE)
            by = y_eval[i:i + batch_size]
            logits = model(bx)
            probs = torch.sigmoid(logits)

            all_probs.extend(probs.cpu().numpy())
            all_targets.extend(by.cpu().numpy())

    probs = np.array(all_probs)
    targets = np.array(all_targets)
    preds = (probs >= threshold).astype(int)

    metrics = {
        "f1_sarc": f1_score(targets, preds, pos_label=1, zero_division=0),
        "precision": precision_score(targets, preds, pos_label=1, zero_division=0),
        "recall": recall_score(targets, preds, pos_label=1, zero_division=0),
        "macro_f1": f1_score(targets, preds, average="macro", zero_division=0),
    }
    return metrics, probs


In [111]:
# 1. Calcul du poids de classe sur train uniquement
num_neg = (y_train == 0).sum().float()
num_pos = (y_train == 1).sum().float()
pos_weight = (num_neg / num_pos).to(DEVICE)
print(f"Poids de la classe sarcastique (pos_weight) : {pos_weight.item():.2f}")

# 2. Définition de la fonction de perte pondérée
criterion_weighted = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# 3. Initialisation d'une instance fraîche du meilleur modèle
set_seed(42)
emb_a = get_embedding("unfrozen")
# Remplacez par le nom de votre meilleur modèle (ex: BiLSTMAttentionClassifier ou HybridSarcasmClassifier)
model_a = BiLSTMAttentionClassifier(embedding_layer=emb_a, hidden_dim=64, attention_dim=64, dropout=0.5).to(DEVICE)

# 4. Entraînement
model_a, best_val_f1_a = train_model(
    model=model_a,
    X_tr=X_train,
    y_tr=y_train,
    X_v=X_val,
    y_v=y_val,
    criterion=criterion_weighted,
    epochs=10,
    batch_size=32,
    lr=1e-3,
    eval_threshold=0.5
)

# 5. Évaluation à seuil standard 0.5
metrics_val_a, _ = evaluate_predictions(model_a, X_val, y_val, threshold=0.5)
print(f"--- (a) Perte Pondérée (Val) ---")
print(f"Val F1-sarc : {metrics_val_a['f1_sarc']:.4f} | Précision : {metrics_val_a['precision']:.4f} | Rappel : {metrics_val_a['recall']:.4f}")


Poids de la classe sarcastique (pos_weight) : 4.79


NameError: name 'get_embedding' is not defined

In [ ]:
def oversample_train_tensors(X_tr, y_tr, seed=42):
    """
    Duplique aléatoirement les exemples positifs de train pour égaler les négatifs.
    """
    torch.manual_seed(seed)
    pos_idx = (y_tr == 1).nonzero(as_tuple=True)[0]
    neg_idx = (y_tr == 0).nonzero(as_tuple=True)[0]

    n_pos = len(pos_idx)
    n_neg = len(neg_idx)

    # Tirer avec remise n_neg indices parmi les positifs
    sampled_pos_idx = pos_idx[torch.randint(0, n_pos, (n_neg,))]

    # Concaténation et mélange
    resampled_idx = torch.cat([neg_idx, sampled_pos_idx])
    perm = torch.randperm(len(resampled_idx))
    final_idx = resampled_idx[perm]

    return X_tr[final_idx], y_tr[final_idx]

# 1. Sur-échantillonnage uniquement sur Train
X_train_resampled, y_train_resampled = oversample_train_tensors(X_train, y_train, seed=42)
print(f"Train après sur-échantillonnage : {len(y_train_resampled)} tweets (50% sarc / 50% non-sarc)")
print(f"Val inchangé : {len(y_val)} tweets")

# 2. Initialisation d'une instance neuve du modèle
set_seed(42)
emb_b = get_embedding("unfrozen")
model_b = BiLSTMAttentionClassifier(embedding_layer=emb_b, hidden_dim=64, attention_dim=64, dropout=0.5).to(DEVICE)

# 3. Entraînement avec perte standard (BCE classique)
criterion_standard = nn.BCEWithLogitsLoss()
model_b, best_val_f1_b = train_model(
    model=model_b,
    X_tr=X_train_resampled,
    y_tr=y_train_resampled,
    X_v=X_val,
    y_v=y_val,
    criterion=criterion_standard,
    epochs=10,
    batch_size=32,
    lr=1e-3,
    eval_threshold=0.5
)

# 4. Évaluation sur Val
metrics_val_b, _ = evaluate_predictions(model_b, X_val, y_val, threshold=0.5)
print(f"--- (b) Sur-échantillonnage (Val) ---")
print(f"Val F1-sarc : {metrics_val_b['f1_sarc']:.4f} | Précision : {metrics_val_b['precision']:.4f} | Rappel : {metrics_val_b['recall']:.4f}")


In [ ]:
# 1. Entraînement avec perte standard
set_seed(42)
emb_c = get_embedding("unfrozen")
model_c = BiLSTMAttentionClassifier(embedding_layer=emb_c, hidden_dim=64, attention_dim=64, dropout=0.5).to(DEVICE)

model_c, _ = train_model(
    model=model_c,
    X_tr=X_train,
    y_tr=y_train,
    X_v=X_val,
    y_v=y_val,
    criterion=nn.BCEWithLogitsLoss(),
    epochs=10,
    batch_size=32,
    lr=1e-3,
    eval_threshold=0.5
)

# 2. Récupération des probabilités continues sur la validation
_, val_probs_c = evaluate_predictions(model_c, X_val, y_val, threshold=0.5)
y_val_np = y_val.cpu().numpy()

# 3. Recherche en grille du seuil optimal maximisant le F1 sarcastique
candidate_thresholds = np.linspace(0.05, 0.95, 91)
best_threshold = 0.5
best_f1_val = -1.0

for th in candidate_thresholds:
    preds = (val_probs_c >= th).astype(int)
    f1 = f1_score(y_val_np, preds, pos_label=1, zero_division=0)
    if f1 > best_f1_val:
        best_f1_val = f1
        best_threshold = th

print(f"--- (c) Seuil Optimisé sur Val ---")
print(f"Seuil optimal trouvé sur Val : {best_threshold:.2f}")

# 4. Métriques sur Val avec le seuil standard 0.5 vs seuil optimal
metrics_std, _ = evaluate_predictions(model_c, X_val, y_val, threshold=0.5)
metrics_opt, _ = evaluate_predictions(model_c, X_val, y_val, threshold=best_threshold)

print(f"• Seuil standard 0.50 : Val F1-sarc = {metrics_std['f1_sarc']:.4f} (Rappel: {metrics_std['recall']:.4f}, Précision: {metrics_std['precision']:.4f})")
print(f"• Seuil optimisé {best_threshold:.2f} : Val F1-sarc = {metrics_opt['f1_sarc']:.4f} (Rappel: {metrics_opt['recall']:.4f}, Précision: {metrics_opt['precision']:.4f})")


La Focal Loss atténue le poids des exemples faciles et bien classés pour forcer le modèle à se concentrer sur les exemples ambigus/difficiles : $$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

In [ ]:
class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        p_t = targets * probs + (1 - targets) * (1 - probs)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_t * ((1 - p_t) ** self.gamma) * bce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss

# 1. Initialisation
set_seed(42)
emb_focal = get_embedding("unfrozen")
model_focal = BiLSTMAttentionClassifier(embedding_layer=emb_focal, hidden_dim=64, attention_dim=64, dropout=0.5).to(DEVICE)

# 2. Entraînement avec Focal Loss (gamma=2.0, alpha=0.25)
criterion_focal = BinaryFocalLoss(alpha=0.25, gamma=2.0)
model_focal, _ = train_model(
    model=model_focal,
    X_tr=X_train,
    y_tr=y_train,
    X_v=X_val,
    y_v=y_val,
    criterion=criterion_focal,
    epochs=10,
    batch_size=32,
    lr=1e-3,
    eval_threshold=0.5
)

# 3. Évaluation sur Val
metrics_val_focal, _ = evaluate_predictions(model_focal, X_val, y_val, threshold=0.5)
print(f"--- Optionnel : Focal Loss (Val) ---")
print(f"Val F1-sarc : {metrics_val_focal['f1_sarc']:.4f} | Précision : {metrics_val_focal['precision']:.4f} | Rappel : {metrics_val_focal['recall']:.4f}")


In [ ]:
# Évaluation sur le test officiel avec les seuils respectifs déterminés sur val
eval_configs = {
    "(a) Perte Pondérée (pos_weight)": (model_a, 0.5),
    "(b) Sur-échantillonnage (train)":  (model_b, 0.5),
    "(c) Seuil Standard (0.50)":        (model_c, 0.5),
    "(c) Seuil Optimisé (val)":         (model_c, best_threshold),
    "(Optionnel) Focal Loss":           (model_focal, 0.5),
}

results_rows = []
for name, (mdl, th) in eval_configs.items():
    v_metrics, _ = evaluate_predictions(mdl, X_val, y_val, threshold=th)
    t_metrics, _ = evaluate_predictions(mdl, X_test, y_test, threshold=th) if X_test is not None else ({}, None)
    
    results_rows.append({
        "Méthode": name,
        "Seuil": f"{th:.2f}",
        "Val F1-sarc": f"{v_metrics['f1_sarc']:.4f}",
        "Val Précision": f"{v_metrics['precision']:.4f}",
        "Val Rappel": f"{v_metrics['recall']:.4f}",
        "Test F1-sarc": f"{t_metrics.get('f1_sarc', 0.0):.4f}",
        "Test Macro-F1": f"{t_metrics.get('macro_f1', 0.0):.4f}",
    })

import pandas as pd
df_summary = pd.DataFrame(results_rows)
print("\n" + "=" * 80)
print("📊 COMPARAISON DES STRATÉGIES DE GESTION DU DÉSÉQUILIBRE (Q6)")
print("=" * 80)
print(df_summary.to_string(index=False))
print("=" * 80)
